In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("SparkExercises") \
    .getOrCreate()

rides_data = [
    ("R001","U001","Hyderabad",12.5,240,"Completed"),
    ("R002","U002","Delhi",8.2,180,"Completed"),
    ("R003","U003","Mumbai",15.0,300,"Cancelled"),
    ("R004","U004","Bangalore",5.5,120,"Completed"),
    ("R005","U005","Hyderabad",20.0,360,"Completed"),
    ("R006","U006","Delhi",25.0,420,"Completed"),
    ("R007","U007","Mumbai",7.5,150,"Completed"),
    ("R008","U008","Bangalore",18.0,330,"Completed"),
    ("R009","U009","Delhi",6.0,140,"Cancelled"),
    ("R010","U010","Hyderabad",10.0,200,"Completed")
]

rides_cols = ["ride_id","user_id","city","distance_km","duration_seconds","status"]
rides_df = spark.createDataFrame(rides_data, rides_cols)

# Surge dataset
surge_data = [
    ("Hyderabad",1.2),
    ("Delhi",1.5),
    ("Mumbai",1.8),
    ("Bangalore",1.3)
]
surge_cols = ["city","surge_multiplier"]
surge_df = spark.createDataFrame(surge_data, surge_cols)

In [3]:

pipeline = rides_df.filter(F.col("status") == "Completed") \
                   .select("ride_id","city","distance_km")


In [4]:
print("Completed rides count:", pipeline.count())

Completed rides count: 8


In [9]:
f_chain = rides_df.filter(F.col("status") == "Completed") \
                   .filter(F.col("distance_km") > 10) \
                   .select("ride_id","city","distance_km")
f_chain.explain(True)

== Parsed Logical Plan ==
'Project ['ride_id, 'city, 'distance_km]
+- Filter (distance_km#3 > cast(10 as double))
   +- Filter (status#5 = Completed)
      +- LogicalRDD [ride_id#0, user_id#1, city#2, distance_km#3, duration_seconds#4L, status#5], false

== Analyzed Logical Plan ==
ride_id: string, city: string, distance_km: double
Project [ride_id#0, city#2, distance_km#3]
+- Filter (distance_km#3 > cast(10 as double))
   +- Filter (status#5 = Completed)
      +- LogicalRDD [ride_id#0, user_id#1, city#2, distance_km#3, duration_seconds#4L, status#5], false

== Optimized Logical Plan ==
Project [ride_id#0, city#2, distance_km#3]
+- Filter ((isnotnull(status#5) AND isnotnull(distance_km#3)) AND ((status#5 = Completed) AND (distance_km#3 > 10.0)))
   +- LogicalRDD [ride_id#0, user_id#1, city#2, distance_km#3, duration_seconds#4L, status#5], false

== Physical Plan ==
*(1) Project [ride_id#0, city#2, distance_km#3]
+- *(1) Filter ((isnotnull(status#5) AND isnotnull(distance_km#3)) AND ((s

In [10]:

joined_then_filtered = rides_df.join(surge_df, on="city", how="inner") \
                                .filter(F.col("distance_km") > 10)
filtered_then_joined = rides_df.filter(F.col("distance_km") > 10) \
                                .join(surge_df, on="city", how="inner")
joined_then_filtered.explain(True)
filtered_then_joined.explain(True)


== Parsed Logical Plan ==
'Filter '`>`('distance_km, 10)
+- Project [city#2, ride_id#0, user_id#1, distance_km#3, duration_seconds#4L, status#5, surge_multiplier#7]
   +- Join Inner, (city#2 = city#6)
      :- LogicalRDD [ride_id#0, user_id#1, city#2, distance_km#3, duration_seconds#4L, status#5], false
      +- LogicalRDD [city#6, surge_multiplier#7], false

== Analyzed Logical Plan ==
city: string, ride_id: string, user_id: string, distance_km: double, duration_seconds: bigint, status: string, surge_multiplier: double
Filter (distance_km#3 > cast(10 as double))
+- Project [city#2, ride_id#0, user_id#1, distance_km#3, duration_seconds#4L, status#5, surge_multiplier#7]
   +- Join Inner, (city#2 = city#6)
      :- LogicalRDD [ride_id#0, user_id#1, city#2, distance_km#3, duration_seconds#4L, status#5], false
      +- LogicalRDD [city#6, surge_multiplier#7], false

== Optimized Logical Plan ==
Project [city#2, ride_id#0, user_id#1, distance_km#3, duration_seconds#4L, status#5, surge_multi

In [26]:

rides_by_city = rides_df.repartition(F.col("city"))
rides_by_city


DataFrame[ride_id: string, user_id: string, city: string, distance_km: double, duration_seconds: bigint, status: string]

In [27]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

In [29]:

no_bc_join = rides_df.join(surge_df, on="city", how="inner")


In [12]:
print("Initial partitions:", rides_df.rdd.getNumPartitions())
rides_4p = rides_df.repartition(4)
print("After repartition(4):", rides_4p.rdd.getNumPartitions())
rides_1p = rides_4p.coalesce(1)
print("After coalesce(1):", rides_1p.rdd.getNumPartitions())

Initial partitions: 2
After repartition(4): 4
After coalesce(1): 1


In [13]:
rides_4p.write.mode("overwrite").parquet("/tmp/parquet_out_4p")
rides_1p.write.mode("overwrite").parquet("/tmp/parquet_out_1p")


In [14]:

rides_by_city = rides_df.repartition(F.col("city"))
rides_by_city.explain(True)


== Parsed Logical Plan ==
'RepartitionByExpression ['city]
+- LogicalRDD [ride_id#0, user_id#1, city#2, distance_km#3, duration_seconds#4L, status#5], false

== Analyzed Logical Plan ==
ride_id: string, user_id: string, city: string, distance_km: double, duration_seconds: bigint, status: string
RepartitionByExpression [city#2]
+- LogicalRDD [ride_id#0, user_id#1, city#2, distance_km#3, duration_seconds#4L, status#5], false

== Optimized Logical Plan ==
RepartitionByExpression [city#2]
+- LogicalRDD [ride_id#0, user_id#1, city#2, distance_km#3, duration_seconds#4L, status#5], false

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Exchange hashpartitioning(city#2, 200), REPARTITION_BY_COL, [plan_id=258]
   +- Scan ExistingRDD[ride_id#0,user_id#1,city#2,distance_km#3,duration_seconds#4L,status#5]



In [16]:

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)  # disable auto-broadcast
no_bc_join = rides_df.join(surge_df, on="city", how="inner")
no_bc_join.explain(True)

filtered_no_bc_join = rides_df.filter(F.col("distance_km") > 10) \
                               .join(surge_df, on="city", how="inner")
filtered_no_bc_join.explain(True)



== Parsed Logical Plan ==
'Join UsingJoin(Inner, [city])
:- LogicalRDD [ride_id#0, user_id#1, city#2, distance_km#3, duration_seconds#4L, status#5], false
+- LogicalRDD [city#6, surge_multiplier#7], false

== Analyzed Logical Plan ==
city: string, ride_id: string, user_id: string, distance_km: double, duration_seconds: bigint, status: string, surge_multiplier: double
Project [city#2, ride_id#0, user_id#1, distance_km#3, duration_seconds#4L, status#5, surge_multiplier#7]
+- Join Inner, (city#2 = city#6)
   :- LogicalRDD [ride_id#0, user_id#1, city#2, distance_km#3, duration_seconds#4L, status#5], false
   +- LogicalRDD [city#6, surge_multiplier#7], false

== Optimized Logical Plan ==
Project [city#2, ride_id#0, user_id#1, distance_km#3, duration_seconds#4L, status#5, surge_multiplier#7]
+- Join Inner, (city#2 = city#6)
   :- Filter isnotnull(city#2)
   :  +- LogicalRDD [ride_id#0, user_id#1, city#2, distance_km#3, duration_seconds#4L, status#5], false
   +- Filter isnotnull(city#6)
    

In [17]:

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10485760)  # reset threshold
bc_join = rides_df.join(F.broadcast(surge_df), on="city", how="inner")
bc_join.explain(True)


== Parsed Logical Plan ==
'Join UsingJoin(Inner, [city])
:- LogicalRDD [ride_id#0, user_id#1, city#2, distance_km#3, duration_seconds#4L, status#5], false
+- ResolvedHint (strategy=broadcast)
   +- LogicalRDD [city#6, surge_multiplier#7], false

== Analyzed Logical Plan ==
city: string, ride_id: string, user_id: string, distance_km: double, duration_seconds: bigint, status: string, surge_multiplier: double
Project [city#2, ride_id#0, user_id#1, distance_km#3, duration_seconds#4L, status#5, surge_multiplier#7]
+- Join Inner, (city#2 = city#6)
   :- LogicalRDD [ride_id#0, user_id#1, city#2, distance_km#3, duration_seconds#4L, status#5], false
   +- ResolvedHint (strategy=broadcast)
      +- LogicalRDD [city#6, surge_multiplier#7], false

== Optimized Logical Plan ==
Project [city#2, ride_id#0, user_id#1, distance_km#3, duration_seconds#4L, status#5, surge_multiplier#7]
+- Join Inner, (city#2 = city#6), rightHint=(strategy=broadcast)
   :- Filter isnotnull(city#2)
   :  +- LogicalRDD [rid

In [18]:

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10485760)  # reset threshold
bc_join = rides_df.join(F.broadcast(surge_df), on="city", how="inner")
bc_join.explain(True)


== Parsed Logical Plan ==
'Join UsingJoin(Inner, [city])
:- LogicalRDD [ride_id#0, user_id#1, city#2, distance_km#3, duration_seconds#4L, status#5], false
+- ResolvedHint (strategy=broadcast)
   +- LogicalRDD [city#6, surge_multiplier#7], false

== Analyzed Logical Plan ==
city: string, ride_id: string, user_id: string, distance_km: double, duration_seconds: bigint, status: string, surge_multiplier: double
Project [city#2, ride_id#0, user_id#1, distance_km#3, duration_seconds#4L, status#5, surge_multiplier#7]
+- Join Inner, (city#2 = city#6)
   :- LogicalRDD [ride_id#0, user_id#1, city#2, distance_km#3, duration_seconds#4L, status#5], false
   +- ResolvedHint (strategy=broadcast)
      +- LogicalRDD [city#6, surge_multiplier#7], false

== Optimized Logical Plan ==
Project [city#2, ride_id#0, user_id#1, distance_km#3, duration_seconds#4L, status#5, surge_multiplier#7]
+- Join Inner, (city#2 = city#6), rightHint=(strategy=broadcast)
   :- Filter isnotnull(city#2)
   :  +- LogicalRDD [rid

In [19]:

long_pipeline = rides_df.filter(F.col("status") == "Completed") \
                        .withColumn("speed_kmph", F.col("distance_km") / (F.col("duration_seconds") / 3600.0)) \
                        .filter(F.col("speed_kmph") > 20) \
                        .withColumn("city_upper", F.upper(F.col("city"))) \
                        .groupBy("city_upper") \
                        .agg(F.count("*").alias("n_rides"),



F.avg("distance_km").alias("avg_distance"),
                             F.avg("speed_kmph").alias("avg_speed")) \
                        .orderBy(F.col("avg_speed").desc()) \
                        .select("city_upper","n_rides","avg_distance","avg_speed")

print("Count:", long_pipeline.count())
long_pipeline.show()
long_pipeline.write.mode("overwrite").parquet("/tmp/long_pipeline_out")



Count: 4
+----------+-------+------------------+------------------+
|city_upper|n_rides|      avg_distance|         avg_speed|
+----------+-------+------------------+------------------+
| HYDERABAD|      3|14.166666666666666|189.16666666666666|
|     DELHI|      2|              16.6| 189.1428571428571|
| BANGALORE|      2|             11.75| 180.6818181818182|
|    MUMBAI|      1|               7.5|             180.0|
+----------+-------+------------------+------------------+



In [22]:

print("count:", long_pipeline.count())   # ACTION → executes the DAG

# 2) show
long_pipeline.show(10, truncate=False)   # ACTION → executes again if not cached

# 3) write
(
    long_pipeline
        .write
               .mode("overwrite")
        .parquet("/tmp/long_pipeline_out")
)

count: 4
+----------+-------+------------------+------------------+
|city_upper|n_rides|avg_distance      |avg_speed         |
+----------+-------+------------------+------------------+
|HYDERABAD |3      |14.166666666666666|189.16666666666666|
|DELHI     |2      |16.6              |189.1428571428571 |
|BANGALORE |2      |11.75             |180.6818181818182 |
|MUMBAI    |1      |7.5               |180.0             |
+----------+-------+------------------+------------------+



In [24]:
long_pipeline_cached = long_pipeline.cache()
long_pipeline_cached.count()
long_pipeline_cached.show()

+----------+-------+------------------+------------------+
|city_upper|n_rides|      avg_distance|         avg_speed|
+----------+-------+------------------+------------------+
| HYDERABAD|      3|14.166666666666666|189.16666666666666|
|     DELHI|      2|              16.6| 189.1428571428571|
| BANGALORE|      2|             11.75| 180.6818181818182|
|    MUMBAI|      1|               7.5|             180.0|
+----------+-------+------------------+------------------+

